<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/2_Extended_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h -nv -O Sources/ach.h

2026-09-05 02:35:14 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h [2787/2787] -> "Sources/ach.h" [1]


Find maximum temperature in each step.

In [3]:
%%writefile Sources/naive-max-diff.cpp

#include "ach.h"
float naive_max_change(
  const thrust::universal_vector<float>& a,
  const thrust::universal_vector<float>& b
)
{
  // allocate a vector to store 'a'-'b'
  thrust::universal_vector<float> unnecessarily_materialized_diff(a.size());

  // compute difference:
  thrust::transform(thrust::device,
                    a.begin(), a.end(), // first input sequence
                    b.begin(),          // second input sequence
                    unnecessarily_materialized_diff.begin(), // store here
                    []__host__ __device__ (float x, float y){
                      return abs(x - y); //
                    });

  // compute max difference
  return thrust::reduce(thrust::device,
                        unnecessarily_materialized_diff.begin(),
                        unnecessarily_materialized_diff.end(),
                        0.0f,
                        thrust::maximum<float>{});

}

int main()
{
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp[] = {{42, 24, 50}, {0, 0, 0}};
  auto transformation = [=]__host__ __device__ (float temp){
    return temp + k*(ambient_temp - temp);
  };

  std::printf("step   max-change\n");
  for(int step = 0; step < 3; step ++){
    thrust::universal_vector<float> &current = temp[step % 2];
    thrust::universal_vector<float> &next = temp[(step +1)%2];

    thrust::transform(thrust::device, current.begin(), current.end(), next.begin(), transformation);
    std::printf("%d   %.2f\n", step, naive_max_change(current, next));
  }
}

Writing Sources/naive-max-diff.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/naive-max-diff.cpp -x cu -arch=native
!/tmp/a.out

step   max-change
0   15.00
1   7.50
2   3.75


**Problem with the above code**
- We started allocating storage for differences.
- We read 2*n floats from both a and b, write n elements back to memory.
- As part of the reduction step, we load n integers.
- Therefore, we have 4*n memory accesses.
- We could implement this in about 2*n memory access with a single for loop.

In [ ]:

# float max_diff = 0.;
# for(int i = 0; i < a.size(); i++){
#     max_diff = std::max(max_diff, std::abs(a[i] - b[i]))
# }


- Now we have a two-fold reduction in amount of memory accesses should result in about two fold speedup. 2x speedup and save space on GPU by avoiding to save the array.

**Iterators**
- A pointer, int* pointer, points to a sequence of integers in memory.
- We can dereference a pointer to get access to the integer it currently points to.
- We can advance pointer with pointer++ to make it point to the next element in the sequence.


In [ ]:
%%writefile Sources/pointer.cpp
#include "ach.h"
int main(){
  std::array<int, 3> a{0, 3, 5};
  int *pointer = a.data();
  std::printf("pointer[0]: %d\n", pointer[0]); // 0
  std::printf("pointer[1]: %d\n", pointer[1]); // 3
}

Writing Sources/pointer.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/pointer.cpp -x cu -arch=native
!/tmp/a.out

pointer[0]: 0
pointer[1]: 3


# Operator overloading:
- In C++ we can define what operators such as *, ++ ,[] do. Concept of iterators build on the top of this idea.



### Simple counter
- We want to create an infinite sequence without allocating a single byte of memory. We will redefine the square brackets [] operator.

Let's first start with iterators:
- Iterators are used to access and iterate through elements of datastructure like vectors, maps, sets by pointing to them.

In [18]:
%%writefile Sources/iterators.cpp
#include<iostream>
#include<vector>

using namespace std;

int main(){
  vector<string> cars = {"Volvo", "BMW", "Ford", "Mazda"};
  // create an iterator called it:
  vector<string>::iterator it;

  // use the iterator to loop through the vector
  // begin() returuns an iterator that points to the first element of the data structure.
  // end() returns an iterator that points to one position after the last element.
  for (it = cars.begin(); it != cars.end(); ++it){
    cout<<*it<<"\n"; // *it is dereferencing it.
  }

  // modify the value:
  it = cars.begin();
  *it = "Tesla";
  cout<<"first element now is: "<< cars[0]<< "\n";

  // The auto keyword: Introduced from C++11
  // instead of vector<string>::iterator = cars.begin(),
  // I can use auto = cars.begin()

  // When you just want to read and not modify elements, use for-each loop:
  for(string car: cars){
    cout<<car<<"\n";
  }

  // when you need to modify, add, remove, or skip elements, use iterators.
  for(auto it = cars.begin(); it!= cars.end();){
    if(*it == "BMW"){
      it = cars.erase(it); // erase?
    } else {
      ++it;
    }
  }

  // print
  for(const string &car : cars){ // why use const with memory address
    cout<<car<<"\n";
  }

  // iterate in reverse:
  for (auto it = cars.rbegin(); it !=cars.rend(); ++it){
    cout<<*it<<"\n";
  }

  return 0;
}


Overwriting Sources/iterators.cpp


In [19]:
!g++ Sources/iterators.cpp -o /tmp/a.out
!/tmp/a.out

Volvo
BMW
Ford
Mazda
first element now is: Tesla
Tesla
BMW
Ford
Mazda
Tesla
Ford
Mazda
Mazda
Ford
Tesla


Let's get back to the counting iterator
- we want to create infinite sequence without using a single byte.
- Trick is to overload [] operator.

In [20]:
%%writefile Sources/counting.cpp
#include "ach.h"

struct counting_iterator
{
  int operator[](int i){
    return i;
  }
};

int main(){
  counting_iterator it;

  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}

Writing Sources/counting.cpp


In [21]:
!nvcc --extended-lambda -o /tmp/a.out Sources/counting.cpp -x cu -arch=native
!/tmp/a.out

it[0]: 0
it[1]: 1


## Simple Transform Iterator
- instead of simple counting, we multiple each input value times 2.

In [27]:
%%writefile Sources/transform.cpp
#include "ach.h"

struct transform_iterator
{
  int *a;
  int operator[](int i){
    return a[i]*2;
  }
};

int main(){
  std::array<int, 3> a{0, 1, 2};
  transform_iterator it{a.data()};

  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}

Overwriting Sources/transform.cpp


In [28]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform.cpp -x cu -arch=native
!/tmp/a.out

it[0]: 0
it[1]: 2


## Simple Zip iterator:
- we can redefine the [] operator to combine two sequences

In [41]:
%%writefile Sources/zip.cpp
#include "ach.h"

struct zip_iterator
{
  int *a;
  int *b;
  std::tuple<int, int> operator[](int i){
    return {a[i], b[i]};
  }
};

int main(){
  std::array<int, 3> a{0, 1, 3};
  std::array<int, 3> b{5, 4, 2};

  zip_iterator it{a.data(), b.data()};
  std::printf("it[0]: (%d, %d)\n", std::get<0>(it[0]), std::get<1>(it[0]));
  std::printf("it[1]: (%d, %d)\n", std::get<0>(it[1]), std::get<1>(it[1]));
}

Overwriting Sources/zip.cpp


In [42]:
!nvcc --extended-lambda -o /tmp/a.out Sources/zip.cpp -x cu -arch=native
!/tmp/a.out

it[0]: (0, 5)
it[1]: (1, 4)


## Combining input iterators:


In [43]:
%%writefile Sources/transform-zip.cpp
#include "ach.h"

struct zip_iterator
{
  int *a;
  int *b;
  std::tuple<int, int> operator[](int i){
    return {a[i], b[i]};
  }
};

struct transform_iterator
{
  zip_iterator zip;
  int operator[](int i){
    auto [a, b] = zip[i];
    return abs(a-b);
  }
};

int main(){
  std::array<int, 3> a{0, 1, 3};
  std::array<int, 3> b{5, 4, 2};

  zip_iterator zip{a.data(), b.data()};
  transform_iterator it{zip};
  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}



Writing Sources/transform-zip.cpp


In [44]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform-zip.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

it[0]: 5
it[1]: 3


## Transform Output Iterator
The concept of iterators is not limited to inputs alone.